# release16_expanded_official_candidate_pool_v1

Purpose: execute the next evidence loop after `release16_nonconvex_checkpoint_training_contract_v1` by expanding the official replay candidate pool beyond the earlier handoff and scoring every available official daily path under the current retention contract.

Governance:
- **Baselines**: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_nonconvex_checkpoint_training_contract_v1`; historical pool comparison `release16_official_replay_candidate_handoff_v1`.
- **Role**: `diagnose` / checkpoint-objective screening. This card does not train, retain, or promote an actor.
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Accepted trade-off**: none for retention; primary gate remains relative return >= +30pp and MaxDD < 20%. A strict 15% MaxDD score is kept as risk context.
- **Changed layer**: `model_selection`, `checkpoint_objective`, `official_replay_candidate_pool`.
- **External reference**: official qlib daily-path replay files from prior checkpoint/BC screens.
- **Hypothesis**: if any existing official replay path already satisfies the 20% drawdown contract across all three walk-forward windows, it can become a promotion-review candidate; otherwise the best failure profile should specify the nonconvex checkpoint/reward objective terms for the next actual training run.
- **Acceptance criteria**: all available official daily paths are audited from source CSVs, no actor retention is allowed unless a full gate exists, and `selection.json` records the best candidate, failure profile, and next mechanism.
- **Closest historical success case**: `release16_nonconvex_checkpoint_training_contract_v1` established a training-ready contract with scope/lift/BC readiness true.
- **Closest historical failure case**: `release16_official_replay_candidate_handoff_v1` found no full-gate candidate in a narrower pool.
- **Why this is not a duplicate**: this run recomputes metrics from every `official_daily_paths/*.csv` across the current repo, not just the narrower curated handoff sources.
- **Comparison-table row key**: `release16_expanded_official_candidate_pool_v1`.


In [1]:
from __future__ import annotations

import json
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
for candidate_root in [ROOT, *ROOT.parents]:
    if (candidate_root / "SPEC.md").exists():
        ROOT = candidate_root
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "runtime/model_selection/card/release16_expanded_official_candidate_pool_v1"
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CONTRACT = ROOT / "runtime/model_selection/card/release16_nonconvex_checkpoint_training_contract_v1/summary.json"
SOURCE_OLD_POOL = ROOT / "runtime/model_selection/card/release16_official_replay_candidate_handoff_v1/summary.json"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
CARD_ROOT = ROOT / "runtime/model_selection/card"
WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
EXPERIMENT = "release16_expanded_official_candidate_pool_v1"

from src.soxl_quant.app.official_candidate_aggregation import (
    OfficialReplayCandidateConfig,
    normalise_official_replay_candidate_frame,
    render_official_replay_candidate_review,
    score_official_replay_candidate_pool,
    score_official_replay_stress_scenarios,
)
from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + chr(10), encoding="utf-8")

source_contract = read_json(SOURCE_CONTRACT)
old_pool = read_json(SOURCE_OLD_POOL)
retention_gate = read_json(RETENTION_GATE)
primary_max_dd = float(source_contract.get("target_max_dd_pct") or retention_gate.get("max_walkforward_drawdown_pct") or 20.0)
primary_config = OfficialReplayCandidateConfig(target_relative_return_pp=30.0, target_max_dd_pct=primary_max_dd)
strict_config = OfficialReplayCandidateConfig(target_relative_return_pp=30.0, target_max_dd_pct=15.0)
print("primary_max_dd", primary_max_dd)
print("source_contract", source_contract["decision"])


primary_max_dd 20.0
source_contract release16_nonconvex_checkpoint_training_contract_ready_no_actor_retention


In [2]:
def parse_official_daily_path(path: Path) -> dict[str, str]:
    source_label = path.relative_to(CARD_ROOT).parts[0]
    stem = path.name
    if "__" in stem:
        window, rest = stem.split("__", 1)
        candidate = rest
    else:
        match = re.search(r"(wf_\d{4}(?:_\d{4})?)", stem)
        window = match.group(1) if match else ""
        candidate = stem.replace(f"{window}_", "") if window else stem
    for suffix in ("_official_daily_path.csv", "_warm_start_official_daily_path.csv", ".csv"):
        if candidate.endswith(suffix):
            candidate = candidate[: -len(suffix)]
            break
    return {"source_label": source_label, "window": window, "candidate": candidate}

rows: list[dict[str, object]] = []
errors: list[dict[str, object]] = []
for path in sorted(CARD_ROOT.glob("*/official_daily_paths/*.csv")):
    meta = parse_official_daily_path(path)
    if meta["window"] not in WINDOWS:
        continue
    try:
        daily = pd.read_csv(path)
        metrics = daily_path_metrics(daily)
    except Exception as exc:  # noqa: BLE001 - diagnostic notebook records source failures.
        errors.append({"source_path": str(path.relative_to(ROOT)), "error": str(exc), **meta})
        continue
    rows.append(
        {
            **meta,
            "source_path": str(path.relative_to(ROOT)),
            "rows": metrics["rows"],
            "start": metrics["start"],
            "end": metrics["end"],
            "return_pct": metrics["return"],
            "buyhold_return_pct": metrics["buy_and_hold_return"],
            "relative_return_pp": metrics["relative_return"],
            "max_dd_pct": metrics["max_dd"],
            "avg_pos": metrics["avg_pos"],
        }
    )

candidate_windows_raw = pd.DataFrame(rows)
read_errors = pd.DataFrame(errors)
if candidate_windows_raw.empty:
    raise RuntimeError("no official daily paths found")
read_errors.to_csv(CARD / "official_daily_path_read_errors.csv", index=False)
candidate_windows_raw.to_csv(CARD / "official_daily_path_window_metrics.csv", index=False)
source_coverage = (
    candidate_windows_raw.groupby("source_label")
    .agg(path_count=("source_path", "count"), candidate_count=("candidate", "nunique"), window_count=("window", "nunique"))
    .reset_index()
    .sort_values(["path_count", "source_label"], ascending=[False, True])
)
source_coverage.to_csv(CARD / "source_coverage.csv", index=False)
print("paths", len(candidate_windows_raw), "sources", candidate_windows_raw["source_label"].nunique(), "source-candidates", candidate_windows_raw[["source_label", "candidate"]].drop_duplicates().shape[0])
source_coverage


paths 123 sources 11 source-candidates 41


,source_label,path_count,candidate_count,window_count
0,cap10_three_fold_checkpoint_metric_replay_v2,12,4,3
1,low_exposure_rally_floor_three_fold_checkpoint...,12,4,3
2,release16_bull_reentry_participation_objective...,12,4,3
3,release16_cb99_rebound_checkpoint_screen_v1,12,4,3
4,release16_failedfold_overlay_checkpoint_screen_v1,12,4,3
5,release16_failedfold_state_head_checkpoint_bc_...,12,4,3
8,release16_rulebase_reversal_checkpoint_metric_...,12,4,3
9,release16_rulebase_reversal_drawdown_upside_ch...,12,4,3
10,return_preserving_bull_participation_three_fol...,12,4,3
7,release16_repaired_bc_warm_start_official_repl...,9,3,3


In [3]:
normalised_frames = []
for source_label, sub in candidate_windows_raw.groupby("source_label", sort=True):
    normalised_frames.append(
        normalise_official_replay_candidate_frame(
            sub,
            source_label=str(source_label),
            mechanism="official_daily_path_pool",
            candidate_col="candidate",
            window_col="window",
            relative_col="relative_return_pp",
            max_dd_col="max_dd_pct",
            avg_pos_col="avg_pos",
            config=primary_config,
        )
    )
primary_windows = pd.concat(normalised_frames, ignore_index=True)
primary_scores = score_official_replay_candidate_pool(primary_windows, config=primary_config)
strict_windows = []
for source_label, sub in candidate_windows_raw.groupby("source_label", sort=True):
    strict_windows.append(
        normalise_official_replay_candidate_frame(
            sub,
            source_label=str(source_label),
            mechanism="official_daily_path_pool",
            candidate_col="candidate",
            window_col="window",
            relative_col="relative_return_pp",
            max_dd_col="max_dd_pct",
            avg_pos_col="avg_pos",
            config=strict_config,
        )
    )
strict_windows = pd.concat(strict_windows, ignore_index=True)
strict_scores = score_official_replay_candidate_pool(strict_windows, config=strict_config)
stress_scores = score_official_replay_stress_scenarios(
    primary_windows,
    stress_scenarios=[
        {"scenario": "light_margin", "relative_haircut_pp": 1.0, "max_dd_add_pct": 0.5},
        {"scenario": "moderate_margin", "relative_haircut_pp": 2.0, "max_dd_add_pct": 1.0},
    ],
    config=primary_config,
)
primary_windows.to_csv(CARD / "candidate_window_metrics_20dd.csv", index=False)
primary_scores.to_csv(CARD / "official_candidate_scores_20dd.csv", index=False)
strict_scores.to_csv(CARD / "official_candidate_scores_15dd.csv", index=False)
stress_scores.to_csv(CARD / "official_candidate_stress_scores.csv", index=False)
primary_scores.head(20)


,candidate_id,source_label,mechanism,candidate,window_count,gate_pass_count,full_gate,robust_gate_pass_count,robust_full_gate,near_miss,min_relative_return,max_max_dd,mean_avg_pos,max_relative_gap,max_dd_gap,min_relative_margin,min_max_dd_margin,total_gate_loss,failure_profile
0,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull65_hold3,3,1,False,1,False,False,14.302062,21.106156,0.236935,15.697938,1.106156,-15.697938,-1.106156,34.477859,joint_tradeoff
1,release16_bull_reentry_participation_objective...,release16_bull_reentry_participation_objective...,official_daily_path_pool,bull70_hold5,3,1,False,1,False,False,11.949062,20.939130,0.210516,18.050938,0.939130,-18.050938,-0.939130,43.140173,joint_tradeoff
2,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,3.406380,16.652151,0.131609,26.593620,0.000000,-26.593620,3.347849,45.173038,trend_capture
3,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,3.406380,16.652151,0.131609,26.593620,0.000000,-26.593620,3.347849,45.173038,trend_capture
4,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,5.082695,20.934612,0.174185,24.917305,0.934612,-24.917305,-0.934612,52.684889,joint_tradeoff
5,release16_rulebase_reversal_checkpoint_metric_...,release16_rulebase_reversal_checkpoint_metric_...,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,5.082695,20.934612,0.174185,24.917305,0.934612,-24.917305,-0.934612,52.684889,joint_tradeoff
6,release16_rulebase_reversal_drawdown_upside_ch...,release16_rulebase_reversal_drawdown_upside_ch...,official_daily_path_pool,bc_initialization,3,1,False,1,False,False,-0.427053,19.326906,0.200377,30.427053,0.000000,-30.427053,0.673094,57.435335,trend_capture
7,release16_rulebase_reversal_drawdown_upside_ch...,release16_rulebase_reversal_drawdown_upside_ch...,official_daily_path_pool,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,-0.427053,19.326906,0.200377,30.427053,0.000000,-30.427053,0.673094,57.435335,trend_capture
8,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,ppo_retained_iter_4,3,1,False,0,False,False,0.920006,20.939975,0.177206,29.079994,0.939975,-29.079994,-0.939975,60.239122,joint_tradeoff
9,release16_failedfold_overlay_checkpoint_screen...,release16_failedfold_overlay_checkpoint_screen_v1,official_daily_path_pool,ppo_selected_best,3,1,False,0,False,False,0.920006,20.939975,0.177206,29.079994,0.939975,-29.079994,-0.939975,60.239122,joint_tradeoff


In [4]:
best = primary_scores.iloc[0].to_dict()
best_id = str(best["candidate_id"])
best_windows = primary_windows[primary_windows["candidate_id"].eq(best_id)].copy()
best_windows = best_windows.merge(
    candidate_windows_raw[["source_label", "candidate", "window", "source_path", "return_pct", "buyhold_return_pct", "rows", "start", "end"]],
    on=["source_label", "candidate", "window"],
    how="left",
)
best_windows.to_csv(CARD / "best_candidate_window_gaps.csv", index=False)
review_md = render_official_replay_candidate_review(primary_scores.head(30))
(CARD / "official_candidate_pool_review.md").write_text(review_md, encoding="utf-8")
print("best", best_id, "full_gate", best["full_gate"], "failure", best["failure_profile"])
best_windows[["window", "relative_return", "max_dd", "avg_pos", "relative_gap", "max_dd_gap", "window_gate_pass", "source_path"]]


best release16_bull_reentry_participation_objective_replay_v1::bull65_hold3 full_gate False failure joint_tradeoff


,window,relative_return,max_dd,avg_pos,relative_gap,max_dd_gap,window_gate_pass,source_path
0,wf_2020,22.281639,21.106156,0.425202,7.718361,1.106156,False,runtime/model_selection/card/release16_bull_re...
1,wf_2022,72.450860,15.072639,0.087837,0.000000,0.000000,True,runtime/model_selection/card/release16_bull_re...
2,wf_2023_2024,14.302062,13.959434,0.197766,15.697938,0.000000,False,runtime/model_selection/card/release16_bull_re...


In [5]:
failure_profile = str(best.get("failure_profile", ""))
objective_terms = [
    "official_replay_relative_return_gap_penalty",
    "official_replay_drawdown_drift_stop_loss",
    "release_slice_fit_guard",
]
if failure_profile in {"joint_tradeoff", "drawdown_control"} or float(best.get("max_dd_gap", 0.0)) > 0.0:
    objective_terms.append("fold_local_drawdown_state_cap_or_penalty")
if failure_profile in {"joint_tradeoff", "trend_capture"} or float(best.get("max_relative_gap", 0.0)) > 0.0:
    objective_terms.append("wf2020_wf2023_trend_participation_floor")
if not bool(source_contract.get("state_head_slice_precondition_gate", False)):
    objective_terms.append("state_head_sparse_slice_precondition_gate")

full_gate_count_20dd = int(primary_scores["full_gate"].sum())
strict_full_gate_count = int(strict_scores["full_gate"].sum())
near_miss_count_20dd = int(primary_scores["near_miss"].sum())
next_mechanism = (
    "rerun_walkforward_promotion_review_for_expanded_official_candidate"
    if full_gate_count_20dd > 0
    else "train_nonconvex_checkpoint_objective_with_relative_floor_drawdown_drift_stop_and_sparse_slice_guard_then_official_three_fold_replay"
)
decision = (
    "expanded_official_candidate_pool_full_gate_review_no_actor_retention"
    if full_gate_count_20dd > 0
    else "expanded_official_candidate_pool_no_full_gate_training_objective_required_no_promotion"
)
summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_nonconvex_checkpoint_training_contract_v1",
    "source_card": "release16_nonconvex_checkpoint_training_contract_v1",
    "source_candidate_pool": "release16_official_replay_candidate_handoff_v1",
    "changed_layer": ["model_selection", "checkpoint_objective", "official_replay_candidate_pool"],
    "target_window": WINDOWS,
    "target_relative_return_pp": 30.0,
    "target_relative_return_by_window_pp": {window: 30.0 for window in WINDOWS},
    "target_max_dd_pct": primary_max_dd,
    "strict_context_max_dd_pct": 15.0,
    "metric_basis": "official_daily_path_recomputed_from_csv",
    "path_count": int(len(candidate_windows_raw)),
    "source_count": int(candidate_windows_raw["source_label"].nunique()),
    "source_candidate_count": int(candidate_windows_raw[["source_label", "candidate"]].drop_duplicates().shape[0]),
    "read_error_count": int(len(read_errors)),
    "full_gate_count_20dd": full_gate_count_20dd,
    "full_gate_count_15dd": strict_full_gate_count,
    "near_miss_count_20dd": near_miss_count_20dd,
    "best_candidate_id": best_id,
    "best_source_label": best.get("source_label"),
    "best_candidate": best.get("candidate"),
    "best_failure_profile": failure_profile,
    "best_total_gate_loss": float(best.get("total_gate_loss", 0.0)),
    "best_gate_pass_count": int(best.get("gate_pass_count", 0)),
    "best_window_count": int(best.get("window_count", 0)),
    "best_min_relative_return_pp": float(best.get("min_relative_return", 0.0)),
    "best_max_dd_pct": float(best.get("max_max_dd", 0.0)),
    "best_max_relative_gap_pp": float(best.get("max_relative_gap", 0.0)),
    "best_max_dd_gap_pct": float(best.get("max_dd_gap", 0.0)),
    "best_mean_avg_pos": float(best.get("mean_avg_pos", 0.0)),
    "metrics": [
        {
            "window": str(row["window"]),
            "split_type": "walk_forward_test",
            "return_pct": float(row["return_pct"]),
            "relative_return_pct": float(row["relative_return"]),
            "max_dd_pct": float(row["max_dd"]),
            "avg_pos": float(row["avg_pos"]),
        }
        for row in best_windows.to_dict(orient="records")
    ],
    "min_wf_relative_return_pp": float(best.get("min_relative_return", 0.0)),
    "worst_wf_max_drawdown_pct": float(best.get("max_max_dd", 0.0)),
    "primary_blocker": "walkforward_relative_return_gap" if full_gate_count_20dd == 0 else "promotion_ready",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "training_contract_ready": False,
    "next_mechanism": next_mechanism,
    "objective_terms": objective_terms,
    "summary_points": [
        f"Expanded official daily-path pool covers {len(candidate_windows_raw)} paths from {candidate_windows_raw['source_label'].nunique()} source cards and {candidate_windows_raw[['source_label', 'candidate']].drop_duplicates().shape[0]} source-candidates.",
        f"No candidate passed all three windows under +30pp / MaxDD < {primary_max_dd:.0f}%" if full_gate_count_20dd == 0 else "At least one candidate reached full 20dd gate and requires promotion review.",
        f"Best candidate is {best_id} with failure_profile={failure_profile}, min relative={float(best.get('min_relative_return', 0.0)):.2f}pp, max DD={float(best.get('max_max_dd', 0.0)):.2f}%.",
        "Next work must train/score a true nonconvex checkpoint objective; reusing existing official replay paths is insufficient for retention." if full_gate_count_20dd == 0 else "Next work is promotion review with strict replay and dashboard evidence, still no automatic actor retention.",
    ],
    "output_paths": {
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "official_daily_path_window_metrics": str((CARD / "official_daily_path_window_metrics.csv").relative_to(ROOT)),
        "candidate_window_metrics_20dd": str((CARD / "candidate_window_metrics_20dd.csv").relative_to(ROOT)),
        "official_candidate_scores_20dd": str((CARD / "official_candidate_scores_20dd.csv").relative_to(ROOT)),
        "official_candidate_scores_15dd": str((CARD / "official_candidate_scores_15dd.csv").relative_to(ROOT)),
        "official_candidate_stress_scores": str((CARD / "official_candidate_stress_scores.csv").relative_to(ROOT)),
        "best_candidate_window_gaps": str((CARD / "best_candidate_window_gaps.csv").relative_to(ROOT)),
        "source_coverage": str((CARD / "source_coverage.csv").relative_to(ROOT)),
        "official_candidate_pool_review": str((CARD / "official_candidate_pool_review.md").relative_to(ROOT)),
    },
}
selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "source_card": summary["source_card"],
    "direct_comparison_baseline": summary["direct_comparison_baseline"],
    "best_candidate_id": best_id,
    "best_failure_profile": failure_profile,
    "full_gate_count_20dd": full_gate_count_20dd,
    "full_gate_count_15dd": strict_full_gate_count,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_mechanism": next_mechanism,
    "objective_terms": objective_terms,
}
write_json(CARD / "summary.json", summary)
write_json(CARD / "selection.json", selection)
summary


{'experiment': 'release16_expanded_official_candidate_pool_v1',
 'created_at': '2026-06-15T00:43:20.099307+00:00',
 'decision': 'expanded_official_candidate_pool_no_full_gate_training_objective_required_no_promotion',
 'role': 'diagnose',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_nonconvex_checkpoint_training_contract_v1',
 'source_card': 'release16_nonconvex_checkpoint_training_contract_v1',
 'source_candidate_pool': 'release16_official_replay_candidate_handoff_v1',
 'changed_layer': ['model_selection',
  'checkpoint_objective',
  'official_replay_candidate_pool'],
 'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_relative_return_pp': 30.0,
 'target_relative_return_by_window_pp': {'wf_2020': 30.0,
  'wf_2022': 30.0,
  'wf_2023_2024': 30.0},
 'target_max_dd_pct': 20.0,
 'strict_context_max_dd_pct': 15.0,
 'metric_basis': 'official_dail

## Conclusion

This notebook is the single-run conclusion for the expanded official replay candidate pool. It recomputes all available official daily-path metrics and scores them against the current retention contract. If `summary.json` reports `full_gate_count_20dd=0`, there is no retained actor candidate here; the next useful step is an actual nonconvex checkpoint objective training/scoring run with relative-return floors, drawdown-drift stop loss, and sparse-slice guard before another official three-fold replay.
